In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

In [2]:
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
nltk.download('wordnet')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [5]:
df= pd.read_csv("Resume Dataset.csv")

In [6]:
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [7]:
df.shape

(2484, 4)

In [8]:
df.columns

Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB


In [10]:
df.isnull().sum()

ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64

In [11]:
#NLP Preprocessing
df["Resume_str"].head()

0             HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1             HR SPECIALIST, US HR OPERATIONS      ...
2             HR DIRECTOR       Summary      Over 2...
3             HR SPECIALIST       Summary    Dedica...
4             HR MANAGER         Skill Highlights  ...
Name: Resume_str, dtype: object

In [12]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\\','',text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text
df['clean_resume']=df['Resume_str'].apply(clean_text)

In [13]:
df.head()

,ID,Resume_str,Resume_html,Category,clean_resume
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr administratormarketing associate\n...
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,hr specialist us hr operations ...
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr director summary over ...
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr specialist summary dedica...
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr manager skill highlights ...


In [14]:
df[['Resume_str','clean_resume']].head()

,Resume_str,clean_resume
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administratormarketing associate\n...
1,"HR SPECIALIST, US HR OPERATIONS ...",hr specialist us hr operations ...
2,HR DIRECTOR Summary Over 2...,hr director summary over ...
3,HR SPECIALIST Summary Dedica...,hr specialist summary dedica...
4,HR MANAGER Skill Highlights ...,hr manager skill highlights ...


In [15]:
df['Resume_str'].head()

0             HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1             HR SPECIALIST, US HR OPERATIONS      ...
2             HR DIRECTOR       Summary      Over 2...
3             HR SPECIALIST       Summary    Dedica...
4             HR MANAGER         Skill Highlights  ...
Name: Resume_str, dtype: object

In [16]:
# stopwords Removal
stop_words = set(stopwords.words('english'))

df['clean_resume'] = df['clean_resume'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop_words])
)

In [17]:
df['clean_resume'].head()

0    hr administratormarketing associate hr adminis...
1    hr specialist us hr operations summary versati...
2    hr director summary years experience recruitin...
3    hr specialist summary dedicated driven dynamic...
4    hr manager skill highlights hr skills hr depar...
Name: clean_resume, dtype: object

In [18]:
#Lemmatization 
lemmatizer = WordNetLemmatizer()
df['clean_resume'] = df['clean_resume'].apply(
    lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()])
)

In [19]:
df['clean_resume'].head()

0    hr administratormarketing associate hr adminis...
1    hr specialist u hr operation summary versatile...
2    hr director summary year experience recruiting...
3    hr specialist summary dedicated driven dynamic...
4    hr manager skill highlight hr skill hr departm...
Name: clean_resume, dtype: object

In [20]:
#feature Representation
tfidf = TfidfVectorizer(max_features=5000)
resume_vectors = tfidf.fit_transform(df['clean_resume'])
print(resume_vectors.shape)

(2484, 5000)


In [21]:
# Similarity Engine    #Ranking Engine
job_description = "Python SQL Machine Learning Data Analysis"

In [22]:
jd_vector = tfidf.transform([job_description])

In [23]:
similarity_scores = cosine_similarity(resume_vectors, jd_vector)

In [24]:
df['similarity_score'] = similarity_scores.flatten()

In [25]:
ranked_resumes = df.sort_values(by='similarity_score',ascending=False)
ranked_resumes[['Category','similarity_score']].head(10)

,Category,similarity_score
1762,ENGINEERING,0.319583
1218,CONSULTANT,0.284897
1142,CONSULTANT,0.267676
2153,BANKING,0.254579
1339,AUTOMOBILE,0.245021
1303,DIGITAL-MEDIA,0.230324
1348,AUTOMOBILE,0.214204
926,AGRICULTURE,0.201002
331,INFORMATION-TECHNOLOGY,0.191309
746,HEALTHCARE,0.190963


In [26]:
#OutPut Layer
#top 10 ranked
ranked_resumes[['Category','similarity_score']].head(10)

,Category,similarity_score
1762,ENGINEERING,0.319583
1218,CONSULTANT,0.284897
1142,CONSULTANT,0.267676
2153,BANKING,0.254579
1339,AUTOMOBILE,0.245021
1303,DIGITAL-MEDIA,0.230324
1348,AUTOMOBILE,0.214204
926,AGRICULTURE,0.201002
331,INFORMATION-TECHNOLOGY,0.191309
746,HEALTHCARE,0.190963


In [27]:
#Match Percentage
ranked_resumes['match_percentage'] = ranked_resumes['similarity_score'] * 100
ranked_resumes[['Category','match_percentage']].head(10)

,Category,match_percentage
1762,ENGINEERING,31.958348
1218,CONSULTANT,28.489690
1142,CONSULTANT,26.767634
2153,BANKING,25.457936
1339,AUTOMOBILE,24.502115
1303,DIGITAL-MEDIA,23.032353
1348,AUTOMOBILE,21.420395
926,AGRICULTURE,20.100220
331,INFORMATION-TECHNOLOGY,19.130889
746,HEALTHCARE,19.096336


In [28]:
import os
print(os.getcwd())

C:\Users\admin\python project
